In [ ]:
import pandas as pd

df = pd.read_parquet("/home/s6019595/liquid/experiments/ablation/20260313_122246/results.parquet")

In [ ]:
def count_params(input_dim, body, out, delegation):                                                                                                                                                                                                      
      """Count parameters per component for a single expert."""                                                                                                                                                                                            
      def _layers(in_dim, neurons):                                                                                                                                                                                                                        
          total, prev = 0, in_dim                                                                                                                                                                                                                        
          for n in neurons:
              total += prev * n + n  # weights + bias
              prev = n
          return total

      body_out = body[-1] if body else input_dim
      bp = _layers(input_dim, body) if body else 0
      op = _layers(body_out, out)
      dp = _layers(body_out, delegation)
      return {"body": bp, "out": op, "del": dp, "total": bp + op + dp}

In [ ]:
count_params(784, body=(128, 64), out=(32, 10), delegation=(32, 10))

In [ ]:
def solve_hidden_dims(total_budget, body_pct, out_pct, del_pct,
                        input_dim=784, body_out=64, n_classes=10, n_models=10):
      """Solve for hidden widths given a param budget and percentages."""
      # params = h * (in + 1 + out) + out  →  h = (budget - out) / (in + 1 + out)
      h_body = int((total_budget * body_pct - body_out) / (input_dim + 1 + body_out))
      h_out  = int((total_budget * out_pct - n_classes) / (body_out + 1 + n_classes))
      h_del  = int((total_budget * del_pct - n_models) / (body_out + 1 + n_models))

      print(f"h_body={h_body}, h_out={h_out}, h_del={h_del}")

      body = (h_body, body_out)
      out = (h_out, n_classes)
      delegation = (h_del, n_models)
      return body, out, delegation

In [ ]:
body, out, deleg = solve_hidden_dims(5000, body_pct=0.10, out_pct=0.40, del_pct=0.50)